# SequenceO1 in 20 Lines: Sketch Attention, Low-Rank Caching, and Why 100K Histories Fit in Your Pocket

Companion notebook for the Medium post **"100,000 Videos in Your Pocket"** (Oct 3, 2026).

SequenceO1 (Guan et al., RecSys '26, arXiv:2609.08443) is deployed at full traffic on Douyin with user histories up to **100,000 interactions**. The trick is *compress-then-reason*:

1. **Sketch Attention** squeezes the raw history into a fixed-size sketch of `k` slots (learnable prototypes, each history token distributes its mass across prototypes).
2. **STCA** (Stacked Target-to-History Cross Attention) reasons over two time scales: the recent **10K suffix** for short-term intent, and the **sketch** for long-term taste.
3. **Low-rank caching**: the sketch is target-agnostic, so it is computed once per user and reused across every candidate, every training instance, and consecutive requests — an `O(1)` cache-hit path in the raw history length `n`.

This notebook builds a toy version in NumPy: a 100K-long synthetic history, a sketch with `k=256` prototypes, and a target-conditioned ranker. No GPU, no downloads.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

N, D, K = 100_000, 64, 256   # raw history, item dim, sketch slots
history = rng.normal(0, 1, size=(N, D)).astype(np.float32)  # 100K item vectors
print(f'raw history: {history.shape}  ({history.nbytes/1e6:.1f} MB fp32)')

## Step 1 — Sketch Attention: 100K items -> k prototype slots

Every history token is softly assigned to the `k` learnable prototypes (softmax over prototype logits), then pooled. The output is a `k x d` sketch whose size **does not depend on `n`** — that is what makes it cacheable.

In [ ]:
prototypes = rng.normal(0, 0.5, size=(K, D)).astype(np.float32)  # learnable in the real model

# assignment: each token distributes mass over prototypes (prototype-wise normalization)
logits = history @ prototypes.T / np.sqrt(D)
logits -= logits.max(axis=1, keepdims=True)
assign = np.exp(logits); assign /= assign.sum(axis=1, keepdims=True)   # (N, K), rows sum to 1

sketch = assign.T @ history        # (K, D) fixed-size user sketch
counts = assign.sum(axis=0)        # how much history each prototype absorbed
print(f'sketch: {sketch.shape}  ({sketch.nbytes/1e3:.0f} KB)  compression: {N//K}x fewer rows')
print(f'prototype load: min {counts.min():.0f}  median {np.median(counts):.0f}  max {counts.max():.0f} items/slot')

## Step 2 — Two time scales: sketch (long-term) + recent suffix (short-term)

A candidate video is scored by attending to (a) the cached sketch and (b) the recent 10K suffix. Both are single-query cross-attention — linear in the attended length, and the sketch term is `O(k)`, not `O(n)`.

In [ ]:
def attend(query, keys):
    s = keys @ query / np.sqrt(query.shape[0])
    s -= s.max(); w = np.exp(s); w /= w.sum()
    return w @ keys

candidate = rng.normal(0, 1, size=D).astype(np.float32)       # the video being ranked
suffix = history[-10_000:]                                     # recent 10K interactions

long_term  = attend(candidate, sketch)     # O(k)  — from the CACHE
short_term = attend(candidate, suffix)     # O(10K) — fresh tail
user_vec = 0.5 * long_term + 0.5 * short_term
score = float(user_vec @ candidate)
print(f'candidate score: {score:.3f}   (sketch read: {K} rows, suffix read: 10,000 rows, raw read avoided: {N:,} rows)')

## Step 3 — The cache math: why O(1) wins at serving time

A ranking request scores ~hundreds of candidates for one user. Without a sketch, every candidate re-reads the 100K history. With it, only the 10K suffix is fresh; the sketch is a cache lookup shared by all candidates.

In [ ]:
CANDIDATES = 300
naive_rows   = CANDIDATES * N
sequenceo1_rows = CANDIDATES * 10_000 + K      # suffix per candidate + one cached sketch
print(f'rows touched per request, naive full-history : {naive_rows:,}')
print(f'rows touched per request, SequenceO1-style   : {sequenceo1_rows:,}')
print(f'reduction: {naive_rows/sequenceo1_rows:.1f}x fewer rows, and the {K}-row sketch is reused across requests too')

## What to take away

- **The sketch is a low-rank summary along the length axis**: `k` prototype slots stand in for `n=100K` items, and its size is independent of `n`, so it can be computed once, cached, and reused — the `O(1)` cache-hit path the name SequenceO1 refers to.
- **Recency is handled separately**: the recent 10K suffix keeps short-term intent exact, while the sketch only has to carry durable taste.
- **Try it**: raise `K` to 1024 and watch prototype load flatten; drop the suffix to 1K and see how much of the score the sketch alone carries. In production, SequenceO1 runs this pattern at full Douyin traffic — the paper reports the cached sketch retains most of the benefit of scaling ranking end-to-end to 100K.